# 🌟 Ejercicios Avanzados: *args y **kwargs - Nivel Intermedio-Avanzado

Este notebook contiene **10 ejercicios complejos** que combinan el uso de `*args` y `**kwargs` con conceptos más avanzados de programación en Python.

## 🎯 Objetivos de aprendizaje:
- Aplicar `*args` y `**kwargs` en sistemas más complejos
- Integrar validaciones, manejo de errores y lógica de negocio
- Trabajar con estructuras de datos complejas y persistencia
- Implementar funciones que simulen sistemas reales
- Combinar múltiples patrones de programación

## 💡 Conceptos que se trabajarán:
- Manejo de archivos con parámetros flexibles
- Sistemas de autenticación y autorización
- Análisis de datos con parámetros configurables
- APIs REST simuladas con parámetros dinámicos
- Sistemas de logs y reportes avanzados
- Validación de datos complejos

## 📊 Nivel de complejidad: **Intermedio a Avanzado**

---

## 🏨 Ejercicio 1: Sistema de Reservas de Hotel Avanzado

**Enunciado:** Implementa un sistema completo de reservas de hotel que maneje múltiples tipos de habitaciones, servicios adicionales, y genere facturas detalladas con descuentos por estancia prolongada.

**Contexto:** Eres el desarrollador de un sistema de reservas para una cadena hotelera internacional. El sistema debe ser flexible para manejar diferentes configuraciones por hotel, temporadas, y políticas de precios.

**Requisitos:**

**Función principal: `procesar_reserva`**
- **Parámetros obligatorios:** `hotel_id`, `fecha_checkin`, `fecha_checkout`, `num_huespedes`
- **`*args`:** IDs de habitaciones a reservar (pueden ser múltiples habitaciones)
- **`**kwargs`:** Servicios adicionales y configuraciones:
  - `desayuno_incluido` (bool, default=False, $15/día/persona)
  - `spa_acceso` (bool, default=False, $25/día/persona)
  - `wifi_premium` (bool, default=False, $8/día)
  - `transporte_aeropuerto` (bool, default=False, $35/trayecto)
  - `seguro_cancelacion` (bool, default=False, 8% del total)
  - `descuento_cliente_frecuente` (float, default=0.0, 0-30%)
  - `codigo_promocional` (str, default=None)
  - `metodo_pago` (str, default="tarjeta", opciones: "tarjeta", "efectivo", "transferencia")

**Datos proporcionados:**
```python
# Base de datos de hoteles
hoteles_db = {
    "HTL001": {
        "nombre": "Grand Plaza Resort",
        "ubicacion": "Cancún, México",
        "habitaciones": {
            "STD": {"nombre": "Estándar", "precio_noche": 120, "capacidad": 2},
            "DLX": {"nombre": "Deluxe", "precio_noche": 180, "capacidad": 3},
            "STI": {"nombre": "Suite", "precio_noche": 350, "capacidad": 4},
            "PNT": {"nombre": "Penthouse", "precio_noche": 650, "capacidad": 6}
        },
        "temporada_alta": ["2024-12-15", "2025-01-15", "2025-07-01", "2025-08-31"],
        "multiplicador_temporada": 1.4
    },
    "HTL002": {
        "nombre": "Business Center Hotel",
        "ubicacion": "Ciudad de México",
        "habitaciones": {
            "STD": {"nombre": "Estándar", "precio_noche": 95, "capacidad": 2},
            "DLX": {"nombre": "Deluxe", "precio_noche": 140, "capacidad": 2},
            "STI": {"nombre": "Suite Ejecutiva", "precio_noche": 280, "capacidad": 4}
        },
        "temporada_alta": ["2025-03-01", "2025-04-30"],
        "multiplicador_temporada": 1.2
    }
}

# Códigos promocionales válidos
codigos_promocion = {
    "SUMMER2024": 15,  # 15% descuento
    "BUSINESS20": 20,  # 20% descuento
    "FAMILY10": 10,    # 10% descuento
    "NEWCLIENT": 25    # 25% descuento (solo nuevos clientes)
}
```

**Funciones adicionales a implementar:**
1. **`validar_disponibilidad`** - Simula verificación de disponibilidad
2. **`calcular_estancia`** - Calcula días de estancia y aplica descuentos por estancia prolongada:
   - 7-13 días: 5% descuento
   - 14-20 días: 10% descuento  
   - 21+ días: 15% descuento
3. **`aplicar_promocion`** - Valida y aplica códigos promocionales
4. **`generar_factura_detallada`** - Genera factura con desglose completo

**Validaciones requeridas:**
- Fechas válidas (checkin antes que checkout)
- Capacidad de habitaciones vs número de huéspedes
- Existencia de hotel y tipos de habitación
- Códigos promocionales válidos
- Combinación de descuentos (máximo 40% total)

**Ejemplo de uso esperado:**
```python
reserva = procesar_reserva(
    "HTL001", "2025-07-15", "2025-07-22", 4,
    "DLX", "STD",  # 2 habitaciones
    desayuno_incluido=True,
    spa_acceso=True,
    codigo_promocional="SUMMER2024",
    descuento_cliente_frecuente=10.0,
    seguro_cancelacion=True,
    metodo_pago="tarjeta"
)
```

**Salida esperada:** Diccionario con detalles completos de la reserva, factura detallada y confirmación.

In [ ]:
# 🚀 EJERCICIO 1: Sistema de Reservas de Hotel Avanzado

from datetime import datetime, timedelta
import json

# Base de datos de hoteles (proporcionada)
hoteles_db = {
    "HTL001": {
        "nombre": "Grand Plaza Resort",
        "ubicacion": "Cancún, México",
        "habitaciones": {
            "STD": {"nombre": "Estándar", "precio_noche": 120, "capacidad": 2},
            "DLX": {"nombre": "Deluxe", "precio_noche": 180, "capacidad": 3},
            "STI": {"nombre": "Suite", "precio_noche": 350, "capacidad": 4},
            "PNT": {"nombre": "Penthouse", "precio_noche": 650, "capacidad": 6}
        },
        "temporada_alta": ["2024-12-15", "2025-01-15", "2025-07-01", "2025-08-31"],
        "multiplicador_temporada": 1.4
    },
    "HTL002": {
        "nombre": "Business Center Hotel",
        "ubicacion": "Ciudad de México",
        "habitaciones": {
            "STD": {"nombre": "Estándar", "precio_noche": 95, "capacidad": 2},
            "DLX": {"nombre": "Deluxe", "precio_noche": 140, "capacidad": 2},
            "STI": {"nombre": "Suite Ejecutiva", "precio_noche": 280, "capacidad": 4}
        },
        "temporada_alta": ["2025-03-01", "2025-04-30"],
        "multiplicador_temporada": 1.2
    }
}

# Códigos promocionales válidos (proporcionados)
codigos_promocion = {
    "SUMMER2024": 15,  # 15% descuento
    "BUSINESS20": 20,  # 20% descuento
    "FAMILY10": 10,    # 10% descuento
    "NEWCLIENT": 25    # 25% descuento (solo nuevos clientes)
}

# TODO: Implementa las funciones auxiliares
def validar_disponibilidad(hotel_id, habitaciones, fecha_checkin, fecha_checkout):
    """Simula validación de disponibilidad de habitaciones"""
    pass

def calcular_estancia(fecha_checkin, fecha_checkout):
    """Calcula días de estancia y aplica descuentos por estancia prolongada"""
    pass

def aplicar_promocion(codigo, total):
    """Valida y aplica códigos promocionales"""
    pass

def generar_factura_detallada(detalles_reserva):
    """Genera factura con desglose completo"""
    pass

# TODO: Implementa la función principal
def procesar_reserva(hotel_id, fecha_checkin, fecha_checkout, num_huespedes, *args, **kwargs):
    """
    Procesa una reserva completa de hotel con servicios adicionales
    
    Args:
        hotel_id: ID del hotel
        fecha_checkin: Fecha de entrada (formato: "YYYY-MM-DD")
        fecha_checkout: Fecha de salida (formato: "YYYY-MM-DD")
        num_huespedes: Número de huéspedes
        *args: IDs de habitaciones a reservar
        **kwargs: Servicios adicionales y configuraciones
    
    Returns:
        dict: Detalles completos de la reserva
    """
    # Tu código aquí
    pass

# TODO: Prueba el sistema con diferentes casos
print("=== PRUEBAS DEL SISTEMA DE RESERVAS ===")

# Caso 1: Reserva compleja con múltiples servicios
try:
    reserva1 = procesar_reserva(
        "HTL001", "2025-07-15", "2025-07-22", 4,
        "DLX", "STD",  # 2 habitaciones
        desayuno_incluido=True,
        spa_acceso=True,
        codigo_promocional="SUMMER2024",
        descuento_cliente_frecuente=10.0,
        seguro_cancelacion=True,
        metodo_pago="tarjeta"
    )
    print("Reserva 1 procesada exitosamente")
except Exception as e:
    print(f"Error en reserva 1: {e}")

# Caso 2: Estancia prolongada con pocos servicios
# Caso 3: Hotel no válido (manejo de errores)
# Caso 4: Sobrecapacidad de habitaciones

print("Implementa el código y ejecuta para probar tu solución")

## 📊 Ejercicio 2: Analizador de Datos Financieros con Configuración Flexible

**Enunciado:** Desarrolla un sistema de análisis financiero que procese múltiples fuentes de datos y genere reportes personalizables con diferentes métricas y formatos de salida.

**Contexto:** Trabajas para una empresa de consultoría financiera que necesita analizar carteras de inversión de diferentes clientes. El sistema debe ser flexible para manejar diferentes tipos de análisis, períodos, y formatos de reporte.

**Requisitos:**

**Función principal: `analizar_cartera`**
- **Parámetros obligatorios:** `cliente_id`, `fecha_inicio`, `fecha_fin`
- **`*args`:** Códigos de instrumentos financieros a analizar (acciones, bonos, ETFs)
- **`**kwargs`:** Configuraciones de análisis:
  - `incluir_dividendos` (bool, default=True)
  - `incluir_splits` (bool, default=True)
  - `moneda_base` (str, default="USD", opciones: "USD", "EUR", "MXN")
  - `benchmark` (str, default="SPY", para comparación)
  - `metricas` (list, default=["retorno", "volatilidad", "sharpe"])
  - `formato_reporte` (str, default="resumen", opciones: "resumen", "detallado", "grafico")
  - `incluir_riesgo` (bool, default=True)
  - `periodo_ventana` (int, default=252, días para cálculos)
  - `exportar_csv` (bool, default=False)
  - `nivel_confianza` (float, default=0.95, para VaR)

**Datos proporcionados:**
```python
# Base de datos de precios históricos (simulada)
precios_historicos = {
    "AAPL": {
        "2025-01-01": 180.5, "2025-01-02": 182.1, "2025-01-03": 179.8,
        "2025-01-04": 185.2, "2025-01-05": 187.6, "2025-01-06": 184.3,
        # ... más datos
    },
    "GOOGL": {
        "2025-01-01": 2450.2, "2025-01-02": 2467.8, "2025-01-03": 2441.5,
        # ... más datos
    },
    "TSLA": {
        "2025-01-01": 238.4, "2025-01-02": 245.7, "2025-01-03": 235.9,
        # ... más datos
    },
    "SPY": {  # Benchmark
        "2025-01-01": 478.2, "2025-01-02": 481.5, "2025-01-03": 476.8,
        # ... más datos
    }
}

# Información de instrumentos
instrumentos_info = {
    "AAPL": {"nombre": "Apple Inc.", "tipo": "accion", "sector": "tecnologia", "dividendo_yield": 0.52},
    "GOOGL": {"nombre": "Alphabet Inc.", "tipo": "accion", "sector": "tecnologia", "dividendo_yield": 0.0},
    "TSLA": {"nombre": "Tesla Inc.", "tipo": "accion", "sector": "automotriz", "dividendo_yield": 0.0},
    "SPY": {"nombre": "SPDR S&P 500 ETF", "tipo": "etf", "sector": "diversificado", "dividendo_yield": 1.3}
}

# Carteras de clientes
carteras_clientes = {
    "CLI001": {
        "nombre": "Inversiones Corporativas SA",
        "tipo": "institucional",
        "tolerancia_riesgo": "moderada",
        "posiciones": {"AAPL": 1000, "GOOGL": 200, "SPY": 500}
    },
    "CLI002": {
        "nombre": "Juan Pérez",
        "tipo": "individual", 
        "tolerancia_riesgo": "agresiva",
        "posiciones": {"TSLA": 150, "AAPL": 300}
    }
}
```

**Funciones adicionales a implementar:**
1. **`calcular_retornos`** - Calcula retornos diarios, semanales, mensuales
2. **`calcular_metricas_riesgo`** - VaR, volatilidad, correlaciones
3. **`comparar_benchmark`** - Alpha, Beta, Tracking Error
4. **`generar_reporte`** - Formatea salida según tipo especificado
5. **`exportar_datos`** - Guarda resultados en CSV si se solicita

**Métricas a calcular:**
- Retorno total y anualizado
- Volatilidad (desviación estándar)
- Ratio de Sharpe
- Máximo drawdown
- VaR (Value at Risk)
- Beta vs benchmark
- Correlación entre activos

**Ejemplo de uso esperado:**
```python
analisis = analizar_cartera(
    "CLI001", "2025-01-01", "2025-01-31",
    "AAPL", "GOOGL", "SPY",
    incluir_dividendos=True,
    moneda_base="USD",
    benchmark="SPY",
    metricas=["retorno", "volatilidad", "sharpe", "var"],
    formato_reporte="detallado",
    incluir_riesgo=True,
    exportar_csv=True
)
```

In [ ]:
# 🚀 EJERCICIO 2: Analizador de Datos Financieros

import csv
import statistics
from datetime import datetime, timedelta

# Datos proporcionados
precios_historicos = {
    "AAPL": {
        "2025-01-01": 180.5, "2025-01-02": 182.1, "2025-01-03": 179.8,
        "2025-01-04": 185.2, "2025-01-05": 187.6, "2025-01-06": 184.3,
        "2025-01-07": 189.1, "2025-01-08": 191.2, "2025-01-09": 188.7,
        "2025-01-10": 192.3, "2025-01-11": 190.5, "2025-01-12": 194.8
    },
    "GOOGL": {
        "2025-01-01": 2450.2, "2025-01-02": 2467.8, "2025-01-03": 2441.5,
        "2025-01-04": 2489.6, "2025-01-05": 2512.3, "2025-01-06": 2498.7,
        "2025-01-07": 2534.1, "2025-01-08": 2547.9, "2025-01-09": 2523.4,
        "2025-01-10": 2561.2, "2025-01-11": 2548.8, "2025-01-12": 2575.6
    },
    "TSLA": {
        "2025-01-01": 238.4, "2025-01-02": 245.7, "2025-01-03": 235.9,
        "2025-01-04": 251.2, "2025-01-05": 248.6, "2025-01-06": 242.8,
        "2025-01-07": 258.3, "2025-01-08": 262.1, "2025-01-09": 254.7,
        "2025-01-10": 267.5, "2025-01-11": 261.9, "2025-01-12": 271.2
    },
    "SPY": {
        "2025-01-01": 478.2, "2025-01-02": 481.5, "2025-01-03": 476.8,
        "2025-01-04": 485.1, "2025-01-05": 488.3, "2025-01-06": 483.9,
        "2025-01-07": 491.7, "2025-01-08": 494.2, "2025-01-09": 489.6,
        "2025-01-10": 496.8, "2025-01-11": 493.4, "2025-01-12": 499.1
    }
}

instrumentos_info = {
    "AAPL": {"nombre": "Apple Inc.", "tipo": "accion", "sector": "tecnologia", "dividendo_yield": 0.52},
    "GOOGL": {"nombre": "Alphabet Inc.", "tipo": "accion", "sector": "tecnologia", "dividendo_yield": 0.0},
    "TSLA": {"nombre": "Tesla Inc.", "tipo": "accion", "sector": "automotriz", "dividendo_yield": 0.0},
    "SPY": {"nombre": "SPDR S&P 500 ETF", "tipo": "etf", "sector": "diversificado", "dividendo_yield": 1.3}
}

carteras_clientes = {
    "CLI001": {
        "nombre": "Inversiones Corporativas SA",
        "tipo": "institucional",
        "tolerancia_riesgo": "moderada",
        "posiciones": {"AAPL": 1000, "GOOGL": 200, "SPY": 500}
    },
    "CLI002": {
        "nombre": "Juan Pérez",
        "tipo": "individual", 
        "tolerancia_riesgo": "agresiva",
        "posiciones": {"TSLA": 150, "AAPL": 300}
    }
}

# TODO: Implementa las funciones auxiliares
def calcular_retornos(precios, tipo="diario"):
    """Calcula retornos diarios, semanales o mensuales"""
    pass

def calcular_metricas_riesgo(retornos, nivel_confianza=0.95):
    """Calcula VaR, volatilidad, correlaciones"""
    pass

def comparar_benchmark(retornos_activo, retornos_benchmark):
    """Calcula Alpha, Beta, Tracking Error"""
    pass

def generar_reporte(datos_analisis, formato="resumen"):
    """Formatea salida según tipo especificado"""
    pass

def exportar_datos(datos, filename):
    """Guarda resultados en CSV"""
    pass

# TODO: Implementa la función principal
def analizar_cartera(cliente_id, fecha_inicio, fecha_fin, *args, **kwargs):
    """
    Analiza cartera de inversión con configuración flexible
    
    Args:
        cliente_id: ID del cliente
        fecha_inicio: Fecha inicio análisis
        fecha_fin: Fecha fin análisis
        *args: Códigos de instrumentos a analizar
        **kwargs: Configuraciones de análisis
    
    Returns:
        dict: Resultados del análisis financiero
    """
    # Configuraciones por defecto
    config = {
        'incluir_dividendos': kwargs.get('incluir_dividendos', True),
        'incluir_splits': kwargs.get('incluir_splits', True),
        'moneda_base': kwargs.get('moneda_base', 'USD'),
        'benchmark': kwargs.get('benchmark', 'SPY'),
        'metricas': kwargs.get('metricas', ['retorno', 'volatilidad', 'sharpe']),
        'formato_reporte': kwargs.get('formato_reporte', 'resumen'),
        'incluir_riesgo': kwargs.get('incluir_riesgo', True),
        'periodo_ventana': kwargs.get('periodo_ventana', 252),
        'exportar_csv': kwargs.get('exportar_csv', False),
        'nivel_confianza': kwargs.get('nivel_confianza', 0.95)
    }
    
    # Tu código aquí
    pass

# TODO: Prueba el sistema
print("=== PRUEBAS DEL ANALIZADOR FINANCIERO ===")

# Caso 1: Análisis completo con exportación
try:
    analisis1 = analizar_cartera(
        "CLI001", "2025-01-01", "2025-01-12",
        "AAPL", "GOOGL", "SPY",
        incluir_dividendos=True,
        moneda_base="USD",
        benchmark="SPY",
        metricas=["retorno", "volatilidad", "sharpe", "var"],
        formato_reporte="detallado",
        incluir_riesgo=True,
        exportar_csv=True
    )
    print("Análisis 1 completado")
except Exception as e:
    print(f"Error en análisis 1: {e}")

print("Implementa el código y ejecuta para probar tu solución")

## 📝 Ejercicio 3: Sistema de Logging y Auditoría Avanzado

**Enunciado:** Desarrolla un sistema de logging empresarial que registre eventos de múltiples aplicaciones con diferentes niveles de severidad, formatos, y destinos de almacenamiento.

**Función principal: `log_evento`**
- **Parámetros obligatorios:** `nivel`, `mensaje`, `aplicacion`
- **`*args`:** Datos adicionales del evento (IDs, valores, códigos)
- **`**kwargs`:** Configuraciones de logging:
  - `timestamp_custom` (datetime, default=None)
  - `usuario_id` (str, default="sistema")
  - `formato` (str, default="json", opciones: "json", "texto", "xml")
  - `destinos` (list, default=["consola"], opciones: "consola", "archivo", "database", "syslog")
  - `incluir_stacktrace` (bool, default=False)
  - `encriptar` (bool, default=False)
  - `nivel_retencion` (int, default=30, días)
  - `tags` (list, default=[])
  - `contexto_extra` (dict, default={})

**Datos proporcionados:**
```python
configuraciones_app = {
    "web_portal": {"prioridad": "alta", "formato_default": "json"},
    "api_service": {"prioridad": "critica", "formato_default": "json"},
    "batch_processor": {"prioridad": "media", "formato_default": "texto"}
}

niveles_severidad = {
    "DEBUG": 10, "INFO": 20, "WARNING": 30, 
    "ERROR": 40, "CRITICAL": 50
}
```

## 🔐 Ejercicio 4: Sistema de Autenticación Multi-Factor

**Enunciado:** Implementa un sistema de autenticación que soporte múltiples métodos de verificación y políticas de seguridad configurables.

**Función principal: `autenticar_usuario`**
- **Parámetros obligatorios:** `username`, `password`
- **`*args`:** Métodos de autenticación adicionales ("sms", "email", "totp", "biometric")
- **`**kwargs`:** Configuraciones de seguridad:
  - `ip_origen` (str, default=None)
  - `dispositivo_confiable` (bool, default=False)
  - `politica_password` (str, default="media", opciones: "baja", "media", "alta")
  - `max_intentos` (int, default=3)
  - `tiempo_bloqueo` (int, default=300, segundos)
  - `session_duration` (int, default=3600, segundos)
  - `require_2fa` (bool, default=True)
  - `log_eventos` (bool, default=True)

**Datos proporcionados:**
```python
usuarios_db = {
    "admin": {"password_hash": "abc123", "role": "admin", "2fa_enabled": True},
    "user1": {"password_hash": "def456", "role": "user", "2fa_enabled": False}
}
```

## 🌐 Ejercicio 5: Simulador de API REST con Middleware

**Enunciado:** Crea un simulador de API REST que maneje diferentes endpoints con middleware personalizable para autenticación, logging, rate limiting y validación.

**Función principal: `procesar_request`**
- **Parámetros obligatorios:** `metodo`, `endpoint`, `datos`
- **`*args`:** Middleware a aplicar (funciones de procesamiento)
- **`**kwargs`:** Configuraciones de request:
  - `headers` (dict, default={})
  - `query_params` (dict, default={})
  - `api_key` (str, default=None)
  - `rate_limit` (int, default=100, requests/min)
  - `timeout` (int, default=30, segundos)
  - `formato_response` (str, default="json")
  - `incluir_metadata` (bool, default=True)
  - `validar_schema` (bool, default=True)

**Datos proporcionados:**
```python
endpoints_config = {
    "/users": {"methods": ["GET", "POST"], "auth_required": True},
    "/products": {"methods": ["GET"], "auth_required": False},
    "/orders": {"methods": ["GET", "POST", "PUT"], "auth_required": True}
}

schemas_validacion = {
    "user": {"required": ["name", "email"], "optional": ["age", "phone"]},
    "product": {"required": ["name", "price"], "optional": ["description", "category"]}
}
```

## 📁 Ejercicio 6: Procesador de Archivos Multi-formato

**Enunciado:** Desarrolla un sistema que procese múltiples tipos de archivos con diferentes transformaciones y validaciones aplicables de forma modular.

**Función principal: `procesar_archivos`**
- **Parámetros obligatorios:** `directorio_origen`, `directorio_destino`
- **`*args`:** Tipos de archivo a procesar ("csv", "json", "xml", "txt", "xlsx")
- **`**kwargs`:** Configuraciones de procesamiento:
  - `transformaciones` (list, default=[], opciones: "limpiar", "normalizar", "validar", "enriquecer")
  - `formato_salida` (str, default="json")
  - `crear_backup` (bool, default=True)
  - `log_errores` (bool, default=True)
  - `batch_size` (int, default=1000)
  - `encoding` (str, default="utf-8")
  - `separador_csv` (str, default=",")
  - `validacion_schema` (dict, default={})
  - `filtros` (dict, default={})

**Datos proporcionados:**
```python
archivos_ejemplo = {
    "ventas.csv": "fecha,producto,cantidad,precio\n2025-01-01,Laptop,2,1500\n",
    "config.json": '{"app": "sistema", "version": "1.0"}',
    "datos.xml": "<root><item id='1'>Producto A</item></root>"
}

transformaciones_disponibles = {
    "limpiar": lambda x: x.strip(),
    "normalizar": lambda x: x.lower(),
    "validar": lambda x: x if len(x) > 0 else None
}
```

## 📈 Ejercicio 7: Generador de Reportes Empresariales

**Enunciado:** Implementa un sistema que genere reportes empresariales complejos con múltiples fuentes de datos, diferentes formatos de salida y análisis estadísticos.

**Función principal: `generar_reporte`**
- **Parámetros obligatorios:** `tipo_reporte`, `periodo`
- **`*args`:** Fuentes de datos a incluir ("ventas", "inventario", "clientes", "finanzas")
- **`**kwargs`:** Configuraciones del reporte:
  - `formato` (str, default="pdf", opciones: "pdf", "excel", "html", "csv")
  - `incluir_graficos` (bool, default=True)
  - `nivel_detalle` (str, default="resumen", opciones: "resumen", "detallado", "completo")
  - `filtros_fecha` (dict, default={})
  - `agrupacion` (str, default="mes", opciones: "dia", "semana", "mes", "trimestre")
  - `metricas_kpi` (list, default=["total", "promedio"])
  - `comparar_periodo_anterior` (bool, default=True)
  - `incluir_tendencias` (bool, default=True)
  - `exportar_datos_raw` (bool, default=False)

**Datos proporcionados:**
```python
datos_ventas = [
    {"fecha": "2025-01-01", "producto": "Laptop", "cantidad": 5, "precio": 1500, "vendedor": "Juan"},
    {"fecha": "2025-01-02", "producto": "Mouse", "cantidad": 20, "precio": 25, "vendedor": "Ana"},
    # ... más datos
]

datos_inventario = [
    {"producto": "Laptop", "stock": 45, "costo": 1200, "proveedor": "TechCorp"},
    {"producto": "Mouse", "stock": 150, "costo": 15, "proveedor": "AccessCorp"},
    # ... más datos
]

plantillas_reporte = {
    "ventas_mensual": {"secciones": ["resumen", "detalle_productos", "performance_vendedores"]},
    "inventario_stock": {"secciones": ["stock_actual", "rotacion", "alertas_minimo"]},
    "financiero": {"secciones": ["ingresos", "costos", "margen", "proyecciones"]}
}
```

## 🌐 Ejercicio 8: Simulador de Red y Monitoreo

**Enunciado:** Crea un simulador de red que monitoree múltiples dispositivos con diferentes protocolos y genere alertas basadas en métricas configurables.

**Función principal: `monitorear_red`**
- **Parámetros obligatorios:** `red_id`, `tiempo_monitoreo`
- **`*args`:** IDs de dispositivos a monitorear
- **`**kwargs`:** Configuraciones de monitoreo:
  - `protocolos` (list, default=["ping", "snmp"], opciones: "ping", "snmp", "ssh", "http")
  - `intervalo_checks` (int, default=60, segundos)
  - `umbral_latencia` (int, default=100, ms)
  - `umbral_cpu` (float, default=80.0, porcentaje)
  - `umbral_memoria` (float, default=85.0, porcentaje)
  - `alertas_email` (bool, default=True)
  - `generar_graficos` (bool, default=False)
  - `log_metricas` (bool, default=True)
  - `tolerancia_fallos` (int, default=3)

**Datos proporcionados:**
```python
dispositivos_red = {
    "RTR001": {"tipo": "router", "ip": "192.168.1.1", "criticidad": "alta"},
    "SWI001": {"tipo": "switch", "ip": "192.168.1.10", "criticidad": "media"},
    "SRV001": {"tipo": "servidor", "ip": "192.168.1.100", "criticidad": "critica"},
    "AP001": {"tipo": "access_point", "ip": "192.168.1.50", "criticidad": "baja"}
}

metricas_simuladas = {
    "RTR001": {"latencia": 45, "cpu": 65, "memoria": 70, "uptime": 99.9},
    "SWI001": {"latencia": 25, "cpu": 30, "memoria": 45, "uptime": 99.8},
    "SRV001": {"latencia": 15, "cpu": 85, "memoria": 90, "uptime": 99.95},
    "AP001": {"latencia": 35, "cpu": 40, "memoria": 55, "uptime": 98.5}
}
```

## 🎯 Ejercicio 9: Sistema de Recomendaciones Inteligente

**Enunciado:** Desarrolla un motor de recomendaciones que analice múltiples factores para sugerir productos, contenido o acciones personalizadas.

**Función principal: `generar_recomendaciones`**
- **Parámetros obligatorios:** `usuario_id`, `contexto`
- **`*args`:** Algoritmos a aplicar ("colaborativo", "contenido", "hibrido", "popular", "trending")
- **`**kwargs`:** Configuraciones del motor:
  - `num_recomendaciones` (int, default=10)
  - `filtros_categoria` (list, default=[])
  - `peso_historial` (float, default=0.7)
  - `peso_similaridad` (float, default=0.3)
  - `incluir_explicaciones` (bool, default=True)
  - `tiempo_ventana` (int, default=30, días)
  - `min_rating` (float, default=3.0)
  - `diversidad` (bool, default=True)
  - `novedad` (bool, default=False)
  - `boost_tendencias` (float, default=1.2)

**Datos proporcionados:**
```python
usuarios_perfil = {
    "USR001": {
        "edad": 28, "genero": "M", "ubicacion": "CDMX",
        "categorias_favoritas": ["tecnologia", "deportes"],
        "historial_compras": ["laptop", "audifonos", "libro_programacion"]
    },
    "USR002": {
        "edad": 34, "genero": "F", "ubicacion": "Guadalajara", 
        "categorias_favoritas": ["hogar", "moda"],
        "historial_compras": ["vestido", "decoracion", "cocina"]
    }
}

productos_catalogo = [
    {"id": "PROD001", "nombre": "Smartphone", "categoria": "tecnologia", "rating": 4.5, "precio": 800},
    {"id": "PROD002", "nombre": "Tablet", "categoria": "tecnologia", "rating": 4.2, "precio": 400},
    {"id": "PROD003", "nombre": "Zapatillas", "categoria": "deportes", "rating": 4.7, "precio": 120},
    {"id": "PROD004", "nombre": "Sofá", "categoria": "hogar", "rating": 4.3, "precio": 1200}
]

interacciones_usuarios = [
    {"usuario": "USR001", "producto": "PROD001", "accion": "view", "timestamp": "2025-01-15"},
    {"usuario": "USR001", "producto": "PROD002", "accion": "cart", "timestamp": "2025-01-16"},
    {"usuario": "USR002", "producto": "PROD004", "accion": "purchase", "timestamp": "2025-01-10"}
]
```

## 💾 Ejercicio 10: Sistema de Backup y Recuperación Empresarial

**Enunciado:** Implementa un sistema completo de backup que maneje múltiples tipos de datos, estrategias de respaldo y políticas de recuperación automática.

**Función principal: `ejecutar_backup`**
- **Parámetros obligatorios:** `nombre_backup`, `tipo_backup`
- **`*args`:** Fuentes de datos a respaldar (directorios, bases de datos, servicios)
- **`**kwargs`:** Configuraciones del backup:
  - `estrategia` (str, default="incremental", opciones: "completo", "incremental", "diferencial")
  - `compresion` (str, default="gzip", opciones: "none", "gzip", "bzip2", "lzma")
  - `encriptacion` (bool, default=True)
  - `destinos` (list, default=["local"], opciones: "local", "cloud", "ftp", "network")
  - `retencion_dias` (int, default=30)
  - `verificar_integridad` (bool, default=True)
  - `notificar_completado` (bool, default=True)
  - `paralelizar` (bool, default=False)
  - `bandwidth_limit` (int, default=None, MB/s)
  - `retry_attempts` (int, default=3)
  - `exclude_patterns` (list, default=[])

**Datos proporcionados:**
```python
fuentes_datos = {
    "/var/www/html": {"tipo": "directorio", "tamaño_gb": 15.5, "prioridad": "alta"},
    "/home/usuarios": {"tipo": "directorio", "tamaño_gb": 250.8, "prioridad": "media"},
    "mysql_prod": {"tipo": "database", "tamaño_gb": 45.2, "prioridad": "critica"},
    "mongodb_logs": {"tipo": "database", "tamaño_gb": 12.3, "prioridad": "baja"},
    "servicio_email": {"tipo": "servicio", "tamaño_gb": 89.1, "prioridad": "alta"}
}

configuraciones_destino = {
    "local": {"path": "/backup/local", "capacidad_gb": 1000, "velocidad_mbps": 150},
    "cloud": {"provider": "AWS S3", "capacidad_gb": "ilimitada", "velocidad_mbps": 50},
    "network": {"server": "backup.empresa.com", "capacidad_gb": 500, "velocidad_mbps": 100}
}

politicas_backup = {
    "critica": {"frecuencia": "diario", "retencion": 90, "verificacion": True},
    "alta": {"frecuencia": "diario", "retencion": 60, "verificacion": True},
    "media": {"frecuencia": "semanal", "retencion": 30, "verificacion": False},
    "baja": {"frecuencia": "mensual", "retencion": 15, "verificacion": False}
}
```

**Funciones adicionales a implementar:**
1. **`calcular_tiempo_estimado`** - Estima duración del backup
2. **`verificar_espacio_disponible`** - Valida capacidad en destinos
3. **`generar_reporte_backup`** - Crea reporte detallado del proceso
4. **`recuperar_desde_backup`** - Función de recuperación automática
5. **`limpiar_backups_antiguos`** - Aplica políticas de retención

**Ejemplo de uso esperado:**
```python
resultado = ejecutar_backup(
    "backup_produccion_2025_01", "completo",
    "/var/www/html", "mysql_prod", "servicio_email",
    estrategia="completo",
    compresion="lzma", 
    encriptacion=True,
    destinos=["local", "cloud"],
    retencion_dias=90,
    verificar_integridad=True,
    paralelizar=True,
    bandwidth_limit=80,
    exclude_patterns=["*.tmp", "*.log"]
)
```

---

## 🎉 Resumen y Conclusiones

¡Felicitaciones! Has completado un conjunto de **10 ejercicios avanzados** que te han permitido aplicar `*args` y `**kwargs` en escenarios empresariales reales.

### 📋 Ejercicios completados:

1. **🏨 Sistema de Reservas de Hotel** - Gestión compleja de servicios y precios
2. **📊 Analizador Financiero** - Procesamiento de datos con métricas configurables  
3. **📝 Sistema de Logging** - Auditoría empresarial con múltiples destinos
4. **🔐 Autenticación Multi-Factor** - Seguridad con políticas flexibles
5. **🌐 API REST Simulada** - Middleware configurable y validaciones
6. **📁 Procesador de Archivos** - Transformaciones modulares multi-formato
7. **📈 Generador de Reportes** - Business Intelligence con múltiples fuentes
8. **🌐 Simulador de Red** - Monitoreo de infraestructura IT
9. **🎯 Sistema de Recomendaciones** - Motor inteligente personalizable
10. **💾 Sistema de Backup** - Recuperación empresarial con múltiples estrategias

### 🚀 Habilidades desarrolladas:

- ✅ **Funciones altamente flexibles** con `*args` y `**kwargs`
- ✅ **Validación compleja de parámetros** y manejo de errores
- ✅ **Configuración dinámica** de sistemas empresariales
- ✅ **Integración de múltiples fuentes** de datos y servicios
- ✅ **Implementación de lógica de negocio** compleja y configurable
- ✅ **Manejo de casos edge** y tolerancia a fallos
- ✅ **Arquitectura modular** y reutilizable
- ✅ **Simulación de sistemas reales** con parámetros empresariales

### 🎯 Próximos pasos recomendados:

1. **Implementa estos sistemas** en proyectos reales
2. **Combina múltiples ejercicios** para crear soluciones más complejas
3. **Agrega persistencia** usando bases de datos reales
4. **Implementa interfaces web** o APIs REST reales
5. **Añade testing unitario** y de integración
6. **Optimiza performance** para grandes volúmenes de datos
7. **Implementa logging real** y monitoreo de producción

### 💡 Conceptos clave dominados:

- **Flexibilidad de parámetros** con `*args` y `**kwargs`
- **Configuración por defecto** inteligente
- **Validación de entrada** robusta
- **Manejo de errores** empresarial
- **Modularidad** y **reutilización** de código
- **Simulación de sistemas** complejos
- **Documentación** clara de APIs

**¡Estás listo para aplicar estos conceptos en proyectos profesionales!** 🚀